[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/ssec/WAF_ML_Tutorial_Part1/blob/main/colab_notebooks/Notebook02_Feature_Engineering.ipynb)

# Notebook 02: Feature Engineering [Colab Version]

### Goal: Understand how and why feature engineering was done on the SEVIR dataset

#### Background

In the last notebook, we showed you some example images of storms in the SEVIR dataset. While some ML approaches are to give ML models every bit of data, meaning we could potentially pass every single pixel in as a predictor, this would be overwhelmingly large (589824 total pixels for just 1 the visible image). So in this notebook we will show you how to engineer some statistics from each image.


#### Reminder of Problem Statement

Before we jump into feature engineering (i.e., calculating statistics for each example), I want to remind you of the ML task we want to accomplish in the paper.

1. Does this image contain a thunderstorm?
2. How many lightning flashes are in this image?

These will be important to consider when choosing the exact values we calculate from the images. Just to be clear, we will assume that we do not have the GOES Lightning Mapper to give us lighting. Instead we will aim to predict what the GOES lightning mapper would measure. While this might not seem initially useful, we could potentially create a climatology of lightning as measured from GOES prior to GOES-16 (i.e., November 2016). Another potential use would be to train a similar model without the radar data, and then use the resulting model on the MODIS sensor, which has been collecting global measurements since 1999 on the NASA TERRA mission. The default images [here](https://worldview.earthdata.nasa.gov/) are MODIS, imagine if we could have lightning flashes with these obs?

#### Step 0: Get the github repo (we need some of the functions there)

The first step with all of these Google Colab notebooks will be to grab the github repo and cd into the notebooks directory.

To run things from the command line, put a ```!``` before your code



In [ ]:
#get the github repo
!git clone https://github.com/ssec/WAF_ML_Tutorial_Part1.git

#cd into the repo so the paths work
import os
os.chdir('/content/WAF_ML_Tutorial_Part1/jupyter_notebooks/')

#### Step 1: Import packages and load data.  
This is basically the same first two steps as the last notebook.


In [ ]:
#needed packages
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

#import some helper functions for our other directory.
import sys
sys.path.insert(1, '../scripts/')
from aux_functions import plot_feature_loc

#plot parameters that I personally like, feel free to make these your own.
import matplotlib
matplotlib.rcParams['axes.facecolor'] = [0.9,0.9,0.9] #makes a grey background to the axis face
matplotlib.rcParams['axes.labelsize'] = 14 #fontsize in pts
matplotlib.rcParams['axes.titlesize'] = 14
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['legend.fontsize'] = 12
matplotlib.rcParams['legend.facecolor'] = 'w'
matplotlib.rcParams['savefig.transparent'] = False

#make default resolution of figures much higher (i.e., High definition)
%config InlineBackend.figure_format = 'retina'

#open an example storm
ds = xr.open_dataset('../datasets/sevir/onestorm.nc')

For fun, lets actually calculate how many pixels are in this dataset. So without re-gridding the number of pixels are:

In [ ]:
print('{} pixels'.format(ds.x.shape[0]**2 + ds.x2.shape[0]**2 + ds.x3.shape[0]**2 + ds.x4.shape[0]**2))

In other words we would have three quarters of 1 million predictors. Which is far too many. Hopefully this illustrates why we choose to extract some statistics from each variable and use those instead.

#### Step 2: Choose Percentiles

While it might be tempting to run with just a single metric from each image, e.g., the mean value, let's think about the data meteorologically.

1. What do the data tell us about the storms?
    - Visible reflectance is larger with thicker storms (i.e., deeper storms)
    - Visible reflectance is low with the surface
    - Water vapor brightness temperature is low with thicker storms, colder surface temperatures
    - Water vapor brightness temperature is larger with thinner storms, larger surface temperatures
    - Clean IR channel is similar to Water Vapor
    - Vertically integrated liquid is larger with stronger updrafts
    
Based on this list of 'interpreting' the data and understanding that in general storms are more likely to contain lightning if they are taller and more intense suggests that we need a range of percentiles, depending on what variable is used. To make things easy, we just calculate all the same percentiles for each image. We choose the [min,1,10,25,median,75,90,99,max] to encompass each one of the above listed storm properties.

In python, the min is the 0th percentile, the median is also the 50th percentile and the max is the 100th percentile. Let's define the desired percentiles here

In [ ]:
desired_percentiles = np.array([0,1,10,25,50,75,90,99,100])

#### Step 3: Calculate percentiles one 1 image

We will now show how the percentiles are calculated for 1 visible image

In [ ]:
#select the first time step (ds_sub means subset of the dataset)
ds_sub = ds.isel(t=0)
#isolate just the visible image (da means DataArray), dont forget the scaling!
da = ds_sub.visible*1e-4
da

numpy has a built in function that will help use calculate the percentiles named nanpercentile

In [ ]:
percentiles = np.nanpercentile(da.values,desired_percentiles,axis=(0,1))
percentiles

#### Step 4: Double check

It is helpful to make sure the calculated statistics make sense on where they come from in the image. So lets visualize it

In [ ]:
#make a figure with size 15 inches in the x, and 10 inches in the y
plt.figure(figsize=(15,10))
#show all x pixels (:) and all y pixels (:) and the first time step, with a Grey colorscale, and the color min 0 and color max 1.
plt.imshow(da,cmap='Greys_r',vmin=0,vmax=1)

#a more advanced plotting function. Dont get bogged down in the details if you dont want
plot_feature_loc(da,plt.gca())

#show us the colorbar
plt.colorbar(label='Visible Reflectance Factor')
#a function that cleans some of the figure up.
plt.tight_layout()

At this point you should take a step back and make sure the locations plotted above make sense. Does the ```min``` show up in a dark spot? Does the ```max``` should up in a bright spot?

Please know that there are alot of pixels in this image. So each percentile likely has more than 1 location in the image. So each time you run the code the labels will be probably in a new spot.

#### Step 5: Run the percentile calculation across more than 1 image

This step is actually quite tricky to do in an efficient manner since we have > 10,000 events each with 49 time steps. For the interested party, please see the scripts titled ```Process_SEVIR.py``` and ```processing_SEVIR_functions.py```

#### Step 6: Look at the results:

After processing your dataset it is always good to give it a look over and make sure there are no spurious data points. Let's take a gander at the Water Vapor file.

All processed files are can be found in the ```datasets``` folder. Each variable gets its own comma separated value (csv) file. The individual files are meant to keep any individual file from becoming > 100 mb. The first column is the datetime, then the next 9 each correspond to the percentiles labeled with ```q_``` and then the number is the percentile, then the last column is a storm report inside the image.

We will use [pandas](https://pandas.pydata.org/), an awesome tool for tabular data to load our csv file

In [ ]:
import pandas as pd

#read csv wants the data path, and we tell it the index is the time, which makes slicing easier later.
df = pd.read_csv('../datasets/sevir/WV_stats_master.csv',index_col=0,low_memory=False)
df.head()

Looking at the first 5 data points, you can see that the numbers look too big to be brightness temperatures (in degC). Thats because there is the scalar offset for this variable as well. This time the authors multiplied by 100, so we need to multiply the percentiles by 10^-2.

In [ ]:
#we dont want to multiply the 'event' column
keys = list(df.keys()[:-1])
df[keys] = df[keys]*1e-2
df.head()

Ah much better. Pandas has some cool built in functions. For example, we can use ```.describe``` to see some summary statistics of each column

In [ ]:
df.describe()

This is helpful in seeing the distribution of each percentile and checking for spurious data. For example, look at the ```min``` row, there is an outlier,
the minimum Tb of q0000 is -183, which is 100 degrees lower than the minimum q001, lets get rid of that erroneous point

In [ ]:
df = df.where(df.q000 > -100)
df.describe()

While tables are great, I am a visual learning, so lets look at these distributions in histogram form

In [ ]:
#make the bins, 50 of them, evenly spaced from -100 degC to 0 degC
bins=np.linspace(-100,0)
#a nice color
b = [126/255,131/255,248/255]

#make a big plot with 9 subplots
fig,axes = plt.subplots(3,3,figsize=(10,10))

#ravel axes
axes = axes.ravel()

#loop over all variables to make things more concise. enumerate makes an iterable and returns a counter (i) and the axis in axes.
for i,ax in enumerate(axes):
    key = keys[i]
    ax.hist(df[key],bins=bins,color=b,edgecolor='k')
    ax.set_title(key)

#add some axis labels
axes[3].set_ylabel('Counts')
axes[7].set_xlabel('Brightness Temperature, [$\degree$C]')

plt.tight_layout()



---



---


**Exercise:** Now for your practice... Next, add cell blocks and comment blocks that do this type of analysis with the other variables (i.e., features). (If you find yourself with extra time at the end, add graphics to double-check your analyses.) I'll get you started on the next one for IR:

In [ ]:
#read csv wants the data path, and we tell it the index is the time, which makes slicing easier later.
df = pd.read_csv('../datasets/sevir/IR_stats_master.csv',index_col=0,low_memory=False)
df.head()